# 第4回 ARIMA・SARIMAとモデル診断

このノートブックは、スライドと同じ順に並んでいます。
コードはすべて完成しているので、上から順に実行して、結果を読んでください。

- 0節：セットアップ（授業の最初に実行する）
- A節：理論をシミュレーションで確かめる
- B節：実データを解析する
- C節：設定を変えて回し直す

今日の問い：定常でない系列を、定常な系列のためのモデル（ARMA）で扱うには、どうすればよいか。

## 0. セットアップ

授業が始まったら、まずこのセルを実行してください。
パッケージのインストールに少し時間がかかります。終わるまで、次のセルは動きません。

In [ ]:
# OSのコードネームを確認（noble = Ubuntu 24.04 など）
system("lsb_release -cs", intern = TRUE)

options(
  repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/noble/2026-09-01"),
  HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
    paste(getRversion(), R.version["platform"], R.version["arch"], R.version["os"]))
)
install.packages(c("tsibble", "tsibbledata", "fable", "feasts", "urca"))
sessionInfo()

## A. 理論をシミュレーションで確かめる

In [ ]:
library(tsibble)
library(tsibbledata)
library(feasts)   # 単位根検定（unitroot_kpss など）。計算には urca パッケージを使う
library(fable)
library(ggtime)   # ACF・残差・予測の図（autoplot、gg_tsresiduals）
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 4.5)
invisible(Sys.setlocale("LC_TIME", "C"))   # 図の月の名前を英語にする

### A-1. 差分をとると、定常になる系列

同じ乱数から、3本の系列を作ります。系列は、第2回と同じく$y_t$と書きます（第1回、第3回の$x_t$と同じものです）。

- AR(1)：$y_t = 0.8\,y_{t-1} + w_t$（定常）
- ランダムウォーク：$y_t = y_{t-1} + w_t$（定常でない）
- ドリフトつきランダムウォーク：$y_t = 0.2 + y_{t-1} + w_t$（定常でない。毎時点0.2ずつ上がる）

In [ ]:
set.seed(1)
n <- 200
w <- rnorm(n)
sim <- tsibble(
  t = rep(1:n, 3),
  series = rep(c("1 AR(1), phi = 0.8", "2 random walk", "3 random walk with drift"), each = n),
  y = c(as.numeric(stats::filter(w, 0.8, method = "recursive")), cumsum(w), cumsum(0.2 + w)),
  key = series, index = t
)
ggplot(sim, aes(t, y)) +
  geom_line() +
  facet_wrap(~series, ncol = 1, scales = "free_y") +
  labs(x = "Time t (step)", y = "Value y_t")

差分（difference）$\nabla y_t = y_t - y_{t-1}$をとります。

In [ ]:
sim_d <- sim |>
  group_by_key() |>
  mutate(dy = difference(y)) |>   # 差分。1時点目は前の値がないので NA になる
  ungroup()
ggplot(sim_d, aes(t, dy)) +
  geom_line(na.rm = TRUE) +
  facet_wrap(~series, ncol = 1) +
  labs(x = "Time t (step)", y = "Difference y_t - y_{t-1}")

下の2本は、差分をとると、同じ高さのまわりを同じ幅で動く系列になります。
差分をとると定常になる系列を、単位根（unit root）をもつ系列と呼びます。

もとの系列と、差分の標本ACFを比べます。

In [ ]:
sim_d |>
  ACF(y, lag_max = 30) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample ACF of y_t")

In [ ]:
sim_d |>
  ACF(dy, lag_max = 30) |>
  autoplot() +
  labs(x = "Lag h (step)", y = "Sample ACF of the difference")

- もとの系列：ランダムウォークのACFは、ほとんど小さくなりません（第1回）。
- 差分：ランダムウォークの差分は、ほぼすべてのラグで破線の中にあります。ホワイトノイズに戻りました。

### A-2. 単位根検定

差分をとるべきかどうかを、検定で決めます。

AR(1)の式$y_t = c + \phi\,y_{t-1} + w_t$の両辺から$y_{t-1}$を引くと、次の回帰の式になります。

$$
\nabla y_t = c + \delta\,y_{t-1} + w_t, \qquad \delta = \phi - 1
$$

$\delta = 0$なら単位根がある（$\phi = 1$）、$\delta < 0$なら定常（$\phi < 1$）です。
そこで、差分を1時点前の値に回帰し、傾きの$t$値を見ます。これをディッキー・フラー検定（Dickey–Fuller test）と呼びます。

In [ ]:
# ディッキー・フラー検定の統計量：差分を、1時点前の値に回帰したときの、傾きの t 値
df_stat <- function(v) {
  dv <- diff(v)               # 差分 y_t - y_{t-1}
  lag1 <- v[-length(v)]       # 1時点前の値 y_{t-1}
  summary(lm(dv ~ lag1))$coefficients[2, 3]
}

帰無仮説（単位根がある）が正しいときに、この$t$値がどう散らばるかを確かめます。
長さ200のランダムウォークを2000本作り、$t$値を2000個とります。

実行する前に、予想してください。ふつうの$t$検定の棄却域（片側5%、$t < -1.645$）に入る割合は、何%でしょうか。

In [ ]:
set.seed(1)
sim_ar1 <- function(n, phi) {           # AR(1)を1本作る（最初の100個は捨てる）。phi = 1 ならランダムウォーク
  v <- stats::filter(rnorm(n + 100), phi, method = "recursive")
  as.numeric(v[-(1:100)])
}
tau <- replicate(2000, df_stat(sim_ar1(200, 1)))
mean(tau < qnorm(0.05))     # ふつうの t 検定の棄却域（-1.645 より小さい）に入った割合
crit <- quantile(tau, 0.05) # 2000個の t 値の、下から5%の点
crit
ggplot(data.frame(tau = tau), aes(tau)) +
  geom_histogram(aes(y = after_stat(density)), bins = 40, fill = "grey60") +
  stat_function(fun = dnorm, xlim = c(-5, 3), colour = "red") +
  geom_vline(xintercept = c(qnorm(0.05), crit), linetype = c("dashed", "solid")) +
  labs(x = "t statistic of the slope", y = "Density")

- 灰色が、2000個の$t$値です。赤い曲線（標準正規分布）より、左にずれています。
- 破線（$-1.645$）より左に入る割合は、5%ではなく、4割を超えます。帰無仮説のもとでは説明変数$y_{t-1}$が定常でないので、いつもの$t$分布が使えません。
- そこで、この検定では、専用の棄却点を使います。実線が、シミュレーションで求めた5%点です。

この棄却点を使って、$\phi$がいろいろな値のAR(1)（長さ200）で、「単位根がある」を棄却できる割合を出します。

In [ ]:
set.seed(1)
phis <- c(0.5, 0.8, 0.9, 0.95, 0.99, 1)
reject <- sapply(phis, function(phi) {
  mean(replicate(1000, df_stat(sim_ar1(200, phi))) < crit)   # 棄却点より小さい割合
})
data.frame(phi = phis, reject = reject)

- $\phi = 1$（帰無仮説が正しい）：棄却する割合は、約5%です。
- $\phi$が1から離れていれば、ほぼ必ず棄却できます。
- $\phi$が1に近い定常な系列は、単位根のある系列と見分けにくくなります。「棄却できない」は、「単位根がある」という意味ではありません。

A-1の3本で試します。

In [ ]:
sim |>
  as_tibble() |>
  group_by(series) |>
  summarise(df = df_stat(y), reject = df_stat(y) < crit)

向きが逆の検定もあります。KPSS検定は、帰無仮説が「定常である」です。p値が小さければ、定常でないと判断します。
`unitroot_ndiffs`は、KPSS検定を使って、定常になるまでに必要な差分の回数を返します。

In [ ]:
sim |> features(y, list(unitroot_kpss, unitroot_ndiffs))
sim_d |> features(dy, unitroot_kpss)

- もとの系列：AR(1)は棄却されず（p値は0.1以上）、下の2本は棄却されます（p値は0.01以下）。必要な差分は、0回、1回、1回です。
- 差分：3本とも棄却されません。

（`kpss_pvalue`は、0.01より小さいときは0.01、0.1より大きいときは0.1と表示されます。）

### A-3. ARIMAモデル

「$d$回差分をとった系列が、ARMA($p$, $q$)に従う」というモデルを、ARIMA($p$, $d$, $q$)と呼びます。

ARIMA(1, 1, 0)の系列を作ります。差分がAR(1)（$\phi = 0.6$）です。

In [ ]:
set.seed(2)
sim_i <- tsibble(t = 1:200, y = cumsum(sim_ar1(200, 0.6)), index = t)
ggplot(sim_i, aes(t, y)) +
  geom_line() +
  labs(x = "Time t (step)", y = "Value y_t")

3つのモデルを当てはめます。

- `right`：ARIMA(1, 1, 0)。作ったときと同じ形
- `wrong`：ARIMA(0, 1, 0)。差分をホワイトノイズとみなす（ランダムウォーク）
- `auto`：次数を自動で選ぶ。$d$はKPSS検定で、$p$と$q$はAICcで選ばれる

AICcは、第3回のAICを、データが少ないとき向けに補正した規準です。使い方はAICと同じで、小さいほうを選びます。

In [ ]:
fit_i <- sim_i |>
  model(right = ARIMA(y ~ 0 + pdq(1, 1, 0)),
        wrong = ARIMA(y ~ 0 + pdq(0, 1, 0)),
        auto  = ARIMA(y))
fit_i
tidy(fit_i) |> select(.model, term, estimate, std.error)
glance(fit_i) |> select(.model, sigma2, AICc)

- `right`のAR係数の推定値は、作ったときの0.6に近い値です。
- `auto`も、ARIMA(1, 1, 0)を選びました。
- `wrong`は、ノイズの分散（`sigma2`）を大きく見積もり、AICcも大きくなります。

### A-4. 残差診断

モデルが合っていれば、残差（1時点先の予測の誤差）はホワイトノイズになるはずです。
残差の図を3つ並べます。上が残差の系列、左下が標本ACF、右下がヒストグラムです。

In [ ]:
fit_i |>
  select(wrong) |>
  gg_tsresiduals(lag_max = 20)

In [ ]:
fit_i |>
  select(right) |>
  gg_tsresiduals(lag_max = 20)

- `wrong`：残差のACFが、ラグ1から何本も破線の外に出ます。モデルが取りこぼした自己相関が、残差に残っています。
- `right`：ほぼすべてのラグで破線の中にあります。

ACFの棒を1本ずつ見る代わりに、ラグ1から$L$までをまとめて検定します。リュング・ボックス検定（Ljung–Box test）です。
帰無仮説は「残差に自己相関がない」です。p値が小さければ、モデルが合っていないと判断します。
`dof`には、推定したARMAの係数の数を入れます。

In [ ]:
augment(fit_i) |>
  filter(.model == "wrong") |>
  features(.innov, ljung_box, lag = 10, dof = 0)
augment(fit_i) |>
  filter(.model == "right") |>
  features(.innov, ljung_box, lag = 10, dof = 1)

### A-5. 予測区間

予測には幅をつけます。「95%の確率で、この中に入る」という範囲を、95%予測区間（prediction interval）と呼びます。

A-1の定常なAR(1)とランダムウォークを、それぞれ作ったときと同じ形のモデルで当てはめ、50時点先まで予測します。

In [ ]:
fit_two <- sim |>
  filter(series != "3 random walk with drift") |>
  model(ar1 = ARIMA(y ~ 1 + pdq(1, 0, 0)),     # 定常なAR(1)
        rw  = ARIMA(y ~ 0 + pdq(0, 1, 0)))     # ランダムウォーク
fc_two <- fit_two |>
  forecast(h = 50) |>
  filter((series == "1 AR(1), phi = 0.8" & .model == "ar1") |
         (series == "2 random walk" & .model == "rw"))
autoplot(fc_two, sim |> filter(series != "3 random walk with drift", t > 100), level = c(80, 95)) +
  facet_wrap(~series, ncol = 1, scales = "free_y") +
  labs(x = "Time t (step)", y = "Value y_t")

95%予測区間の半分の幅を、1、10、50時点先で比べます。

In [ ]:
iv <- hilo(fc_two$y, 95)
fc_two |>
  as_tibble() |>
  mutate(h = t - 200, half_width = (iv$upper - iv$lower) / 2) |>
  filter(h %in% c(1, 10, 50)) |>
  select(series, h, half_width)

- 定常なAR(1)：予測は平均に戻り、幅は一定の値に落ち着きます。
- ランダムウォーク：予測は最後の値のままで、幅は$\sqrt{h}$に比例して広がり続けます。

差分をとるかどうかで、遠い先の予測の幅がまったく変わります。

95%予測区間に、本当に95%が入るかを確かめます。
長さ120のランダムウォークを2000本作り、最初の100個から先の20個を予測します。区間は2通りです。

- `widening`：$y_T \pm 1.96\,\hat{\sigma}\sqrt{h}$（幅を$\sqrt{h}$で広げる）
- `fixed`：$y_T \pm 1.96\,\hat{\sigma}$（幅を広げない）

In [ ]:
set.seed(1)
cover <- replicate(2000, {
  v <- cumsum(rnorm(120))
  obs <- v[1:100]; future <- v[101:120]
  s <- sd(diff(obs))                        # ノイズの標準偏差の推定値
  err <- abs(future - obs[100])             # 最後の値で予測したときの、外れた幅
  c(widening = err <= 1.96 * s * sqrt(1:20), fixed = err <= 1.96 * s)
})
cov_tab <- data.frame(h = 1:20,
                      widening = unname(rowMeans(cover[1:20, ])),
                      fixed = unname(rowMeans(cover[21:40, ])))
cov_tab[c(1, 2, 5, 10, 20), ]

幅を広げた区間は、どの$h$でも約95%が入ります。広げない区間は、先に行くほど入らなくなります。

### A-6. 季節差分と季節ARIMA

月ごとの系列を作ります。長さは240か月（20年）です。直線の傾向と、12か月でくり返す上下があります。

In [ ]:
set.seed(3)
n_s <- 240
theta <- -0.4; Theta <- -0.6                  # MAの係数と、季節MAの係数
w <- rnorm(n_s + 13)
e <- w[14:(n_s + 13)] + theta * w[13:(n_s + 12)] +
  Theta * w[2:(n_s + 1)] + theta * Theta * w[1:n_s]      # (1 + theta B)(1 + Theta B^12) w_t
z <- diffinv(diffinv(e, lag = 12), lag = 1)[-(1:13)]     # 季節差分と差分を、足し戻す
shape <- c(-8, -10, -4, 0, 3, 8, 12, 10, 4, -2, -7, -6)  # 12か月の上下（合計は0）
sim_s <- tsibble(month = yearmonth("2001 Jan") + 0:(n_s - 1),
                 y = 100 + 0.5 * (1:n_s) + rep(shape, length.out = n_s) + z,
                 index = month)
ggplot(sim_s, aes(month, y)) +
  geom_line() +
  labs(x = "Month", y = "Value y_t")

2種類の差分をとります。

- 差分：$\nabla y_t = y_t - y_{t-1}$（1か月前との差）
- 季節差分（seasonal difference）：$\nabla_{12}\,y_t = y_t - y_{t-12}$（1年前の同じ月との差）

In [ ]:
sim_s <- sim_s |>
  mutate(d1 = difference(y),                          # 差分だけ
         d12 = difference(y, lag = 12),               # 季節差分だけ
         d1_d12 = difference(difference(y, lag = 12)))   # 両方
long <- bind_rows(
  as_tibble(sim_s) |> transmute(month, panel = "1 y", value = y),
  as_tibble(sim_s) |> transmute(month, panel = "2 difference", value = d1),
  as_tibble(sim_s) |> transmute(month, panel = "3 seasonal difference", value = d12),
  as_tibble(sim_s) |> transmute(month, panel = "4 both", value = d1_d12))
ggplot(long, aes(month, value)) +
  geom_line(na.rm = TRUE) +
  facet_wrap(~panel, ncol = 1, scales = "free_y") +
  labs(x = "Month", y = NULL)

- 差分だけ：傾向は消えますが、12か月の上下が残ります。
- 季節差分だけ：12か月の上下は消えますが、高さがゆっくり動きます。
- 両方：同じ高さのまわりを、同じ幅で動く系列になります。

両方の差分をとった系列の標本ACFに、理論値（赤い点）を重ねます。
この系列は、差分をとったあとが「MA(1)と、12か月ごとのMA(1)のかけ算」になるように作ってあります。
理論上、ACFが0でないのは、ラグ1、11、12、13だけです。

In [ ]:
theory <- data.frame(lag = 1:36,
                     acf = ARMAacf(ma = c(theta, rep(0, 10), Theta, theta * Theta), lag.max = 36)[-1])
sim_s |>
  ACF(d1_d12, lag_max = 36) |>
  autoplot() +
  geom_point(data = theory, aes(lag, acf), colour = "red", size = 2) +
  labs(x = "Lag h (month)", y = "ACF (bar: sample, red: theory)")

差分とARMAを、季節の周期でもう一度かけたモデルを、季節ARIMA（seasonal ARIMA, SARIMA）と呼びます。
ARIMA($p$, $d$, $q$)($P$, $D$, $Q$)$_{12}$と書きます。後ろのかっこが、12か月ごとの部分です。

作ったときと同じ形、ARIMA(0, 1, 1)(0, 1, 1)$_{12}$を当てはめます。

In [ ]:
fit_s <- sim_s |>
  model(sarima = ARIMA(y ~ 0 + pdq(0, 1, 1) + PDQ(0, 1, 1)))
fit_s
tidy(fit_s) |> select(term, estimate, std.error)
augment(fit_s) |> features(.innov, ljung_box, lag = 24, dof = 2)

`ma1`（$\theta$）と`sma1`（$\Theta$）の推定値は、作ったときの$-0.4$と$-0.6$に近い値です。残差に自己相関は残っていません。

3年先まで予測します。

In [ ]:
fit_s |>
  forecast(h = 36) |>
  autoplot(sim_s |> filter(month >= yearmonth("2015 Jan")), level = c(80, 95)) +
  labs(x = "Month", y = "Value y_t")

予測は、傾向と12か月の上下の両方を引き継ぎます。幅は、先に行くほど広がります。

第2回のベンチマークは、ARIMAの特別な場合です。予測の値が一致することを確かめます。

- ナイーブ法 ＝ ARIMA(0, 1, 0)：差分がホワイトノイズ
- 季節ナイーブ法 ＝ ARIMA(0, 0, 0)(0, 1, 0)$_{12}$：季節差分がホワイトノイズ

In [ ]:
fc_b <- sim_s |>
  model(naive    = NAIVE(y),
        arima010 = ARIMA(y ~ 0 + pdq(0, 1, 0) + PDQ(0, 0, 0)),
        snaive   = SNAIVE(y),
        sarima   = ARIMA(y ~ 0 + pdq(0, 0, 0) + PDQ(0, 1, 0))) |>
  forecast(h = 12) |>
  as_tibble()
c(naive  = max(abs(fc_b$.mean[fc_b$.model == "naive"]  - fc_b$.mean[fc_b$.model == "arima010"])),
  snaive = max(abs(fc_b$.mean[fc_b$.model == "snaive"] - fc_b$.mean[fc_b$.model == "sarima"])))

差の最大値は、どちらも0です。

## B. 実データを解析する

2つのデータを使います。

- `AirPassengers`：国際線の旅客数（月ごと、Rに付属）
- `global_economy`：各国の経済指標（年ごと、`tsibbledata`パッケージ）。国内総生産（GDP）を使う

### B-0. 設定

このセルの値を変えると、B節の解析がすべて変わります。まずはこのまま実行してください。

In [ ]:
# ここを変えると、以下の解析がすべて変わる
series <- "AirPassengers"   # 系列："AirPassengers"（月ごとの旅客数）か、国名（その国のGDP）
                            #   国名："Japan" "United States" "China" "India"
                            #         "Australia" "United Kingdom" "France" "Brazil"
d      <- 1                 # 差分の回数：AirPassengers は 0 か 1、国のGDPは 1 か 2
D      <- 1                 # 季節差分の回数：0 か 1（国のGDPでは使われない）
level  <- 95                # 予測区間の水準（%）：80, 90, 95, 99 のどれか

### B-1. データを確かめる

In [ ]:
AirPassengers |> as_tsibble() |> glimpse()
range(time(AirPassengers))
glimpse(global_economy)
range(global_economy$Year)

- `AirPassengers`：1949年1月から1960年12月までの144か月。値は、月ごとの旅客数（千人）です。
- `global_economy`：1行は、1つの国の1年です。1960年から2017年まで。`GDP`が国内総生産（米ドル）です。

### B-2. 前処理

設定で選んだ系列を取り出し、時点を`time`、値を`y`という名前にそろえます。
あわせて、季節の周期`m`（月ごとなら12、年ごとなら1）と、テストデータの長さ`h`を決めます。

In [ ]:
if (series == "AirPassengers") {
  dat <- as_tsibble(AirPassengers) |>
    rename(time = index, y = value)              # 旅客数（千人）
  m <- 12; h <- 12                               # 周期は12か月。最後の12か月をテストデータにする
  d_ok <- 0:1                                    # この系列で選べる d
  y_lab <- "Passengers (thousands)"; t_lab <- "Month"
} else {
  dat <- global_economy |>
    filter(Country == series) |>
    as_tibble() |>
    transmute(time = Year, y = GDP / 1e9) |>     # GDP（10億米ドル）
    as_tsibble(index = time)
  m <- 1; h <- 5                                 # 季節はない。最後の5年をテストデータにする
  d_ok <- 1:2                                    # この系列で選べる d
  y_lab <- "GDP (billion USD)"; t_lab <- "Year"
}
n <- nrow(dat)
stopifnot("series の名前が、選べる値のどれとも一致しない" = n > 0,
          "この系列では選べない d になっている（設定セルのコメントを見る）" = d %in% d_ok)
interval(dat)
c(n = n, missing = sum(is.na(dat$y)), gaps = has_gaps(dat)$.gaps)

間隔が一定で、欠測も行の抜けもないことを確かめました。

In [ ]:
ggplot(dat, aes(time, y)) +
  geom_line() +
  labs(x = t_lab, y = y_lab, title = series)

### B-3. 対数と差分

上がり下がりの幅が、水準に比例して大きくなる系列は、対数をとると幅がそろいます。
対数をとってから、設定で決めた回数だけ、季節差分と差分をとります。

In [ ]:
dat <- dat |> mutate(log_y = log(y), z = log_y)
if (D == 1 && m > 1) dat <- dat |> mutate(z = difference(z, lag = m))   # 季節差分
if (d >= 1) dat <- dat |> mutate(z = difference(z))                     # 差分
if (d == 2) dat <- dat |> mutate(z = difference(z))                     # 差分（2回目）
long <- bind_rows(
  as_tibble(dat) |> transmute(time, panel = "1 log(y)", value = log_y),
  as_tibble(dat) |> transmute(time, panel = "2 log(y) after differencing", value = z))
ggplot(long, aes(time, value)) +
  geom_line(na.rm = TRUE) +
  facet_wrap(~panel, ncol = 1, scales = "free_y") +
  labs(x = t_lab, y = NULL)

In [ ]:
lag_max <- if (m > 1) 3 * m else 15
dat |>
  ACF(z, lag_max = lag_max) |>
  autoplot() +
  labs(x = "Lag h", y = "Sample ACF after differencing")

KPSS検定で、定常とみなせるかを確かめます。1行目が対数をとった系列、2行目が差分をとったあとの系列です。
`unitroot_nsdiffs`は必要な季節差分の回数、`unitroot_ndiffs`は必要な差分の回数です。

In [ ]:
kpss <- bind_rows(dat |> features(log_y, unitroot_kpss),
                  dat |> features(z, unitroot_kpss)) |>
  mutate(series = c("log(y)", "after differencing"), .before = 1)
kpss
dat |> features(log_y, list(unitroot_nsdiffs, unitroot_ndiffs))

### B-4. モデルを当てはめる

最後の`h`時点をテストデータにし、それより前を訓練データにします。

ARIMAモデルは、差分の回数（`d`と`D`）を設定の値に固定し、残りの次数（$p, q \le 2$、$P, Q \le 1$）をAICcで選びます。
比べる相手は、第2回の3つのベンチマークです。

In [ ]:
train <- dat |> filter(row_number() <= n - h)
fit <- train |>
  model(arima  = ARIMA(log(y) ~ pdq(0:2, d, 0:2) + PDQ(0:1, D, 0:1)),
        mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = RW(y ~ lag(m)))     # 季節ナイーブ法。m = 1 のときは、ナイーブ法と同じになる
fit |> select(arima) |> report()

### B-5. 残差診断

In [ ]:
fit |>
  select(arima) |>
  gg_tsresiduals(lag_max = lag_max)

In [ ]:
# 推定したARMAの係数の数（定数項は数えない）
K <- sum(tidy(fit)$.model == "arima" & tidy(fit)$term != "constant")
lb_lag <- if (m > 1) 2 * m else 10
lb <- augment(fit) |>
  filter(.model == "arima") |>
  features(.innov, ljung_box, lag = lb_lag, dof = K)
lb

### B-6. 予測区間

In [ ]:
fc <- fit |> forecast(h = h)
fc |>
  filter(.model == "arima") |>
  autoplot(dat |> filter(row_number() > n - 5 * h), level = level) +
  labs(x = t_lab, y = y_lab)

黒い線が本当の値、青い線が予測、帯が予測区間です。
テストデータの`h`個のうち、何個が区間に入ったかを数えます。

In [ ]:
fc_arima <- fc |> filter(.model == "arima")
iv <- hilo(fc_arima$y, level)
test <- dat |> filter(row_number() > n - h)
pi_tab <- data.frame(time = test$time, y = test$y, forecast = fc_arima$.mean,
                     lower = iv$lower, upper = iv$upper,
                     inside = test$y >= iv$lower & test$y <= iv$upper)
pi_tab
c(inside = sum(pi_tab$inside), h = h)

### B-7. 予測を比べる

In [ ]:
acc <- accuracy(fc, dat) |> select(.model, RMSE, MAE)
acc

### B-8. 時系列交差検証

訓練データを`h`時点ずつ伸ばしながら、同じ評価を6回くり返します。ARIMAの次数は、そのたびに選び直します。

In [ ]:
acc_cv <- dat |>
  filter(row_number() <= n - h) |>
  stretch_tsibble(.init = n - 6 * h, .step = h) |>
  model(arima  = ARIMA(log(y) ~ pdq(0:2, d, 0:2) + PDQ(0:1, D, 0:1)),
        mean   = MEAN(y),
        naive  = NAIVE(y),
        snaive = RW(y ~ lag(m))) |>
  forecast(h = h) |>
  accuracy(dat) |>
  select(.model, RMSE, MAE)
acc_cv

読み方：

- B-3：差分をとったあとの系列は、同じ高さのまわりを動いていますか。KPSS検定のp値は、0.05より大きいですか。
- B-5：残差のACFは破線の中にありますか。リュング・ボックス検定のp値（`lb_pvalue`）が0.05より小さければ、モデルが取りこぼした自己相関があります。
- B-6：予測区間は、先に行くほど広がりますか。本当の値は、区間に入っていますか。
- B-7、B-8：ARIMAは、ベンチマークより良いですか。1回の評価と6回の評価で、結論は同じですか。

## C. 設定を変えて回し直す

B-0の設定セルを書き替えて、B節を上からすべて実行し直してください
（Colabでは、設定セルを選んで、メニューの［ランタイム］→［以降のセルを実行］）。

変えられる値：

| 変数 | 値 |
| --- | --- |
| `series` | `"AirPassengers"`、または国名 `"Japan"` `"United States"` `"China"` `"India"` `"Australia"` `"United Kingdom"` `"France"` `"Brazil"` |
| `d` | `AirPassengers`では 0、1。国のGDPでは 1、2 |
| `D` | 0、1（国のGDPでは使われない） |
| `level` | 80、90、95、99 |

読むときの観点：

1. `AirPassengers`で、季節差分をやめる（`D <- 0`）と、差分後のACF、残差診断、予測の誤差はどう変わるか。
2. `AirPassengers`で、差分をやめる（`d <- 0`、`D <- 1`）と、選ばれるモデルと予測区間の広がり方はどう変わるか。
3. 系列を国のGDPにすると、差分を1回とったあとの系列は、KPSS検定で定常とみなせるか。みなせない国は、図のどこが理由か。差分を2回にすると、予測区間はどう変わるか。
4. ARIMAがナイーブ法に負ける設定はあるか。そのとき、テストデータの期間に何が起きているか。
5. `level`を80にすると、区間に入る個数はどう変わるか。

正解は1つではありません。自分の設定で出た結果を、最初の設定（AirPassengers、`d = 1`、`D = 1`、95%）の結果と比べてください。